# 01 - Check downloads
Confirms every file is where the notebooks expect it. Run this first and send the output if anything says MISSING.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from paths import *
print("repo root:", ROOT)

In [ ]:
import pandas as pd

def status(label, ok, extra=""):
    print(f"{'OK     ' if ok else 'MISSING'} {label} {extra}")

html_dir = find_dir(CODENET_DIR, "problem_descriptions")
n_html = len(list(html_dir.rglob("*.html"))) if html_dir else 0
status("CodeNet problem_descriptions", html_dir is not None, f"-> {html_dir} ({n_html} html files)")

plist = find_file(CODENET_DIR, "problem_list.csv")
status("CodeNet problem_list.csv", plist is not None, f"-> {plist}")
if plist:
    display(pd.read_csv(plist).head(3))

emb_csv = find_embeddings_csv()
status("AIGCodeSet embeddings csv", emb_csv is not None,
       f"-> {emb_csv} ({emb_csv.stat().st_size/1e6:.0f} MB)" if emb_csv else "")
if emb_csv:
    cols = pd.read_csv(emb_csv, nrows=2).columns.tolist()
    print("   embeddings csv columns:", cols)

In [ ]:
for s in ["train", "val", "test"]:
    p = PROCESSED / f"{s}.parquet"
    status(f"split {s}", p.exists())
    if p.exists():
        df = pd.read_parquet(p)
        print("   rows:", len(df))
        print("   columns:", df.columns.tolist())

p = INTERIM / "problems.parquet"
status("data/interim/problems.parquet", p.exists())
if p.exists():
    pr = pd.read_parquet(p)
    print("   rows:", len(pr), "| columns:", pr.columns.tolist())
    display(pr.head(2))

In [ ]:
pred_files = sorted(RESULTS.rglob("preds_*.parquet"))
status("prediction files from Nour", len(pred_files) > 0, f"({len(pred_files)} found)")
for f in pred_files:
    d = pd.read_parquet(f)
    print("  ", f.relative_to(ROOT), "| rows:", len(d), "| columns:", d.columns.tolist())

for f in ["features_train.parquet", "features_val.parquet", "features_test.parquet"]:
    status(f"features/{f}", (FEATURES / f).exists())